In [0]:
%sql



In [0]:
data = [
    (1, "Alice", 1000),
    (2, "Bob", 2000),
    (3, "Charlie", 3000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)
display(df)
df.show()
#df.write.format("delta").save("/tmp/delta/people")
#display(dbutils.fs.ls("/tmp/delta/people"))
#display(spark.read.format("delta").load("/tmp/delta/people"))

In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('workspace.default.people')

In [0]:
%sql

describe history workspace.default.people

In [0]:
df_uc = spark.read.table("people")
df_uc.show()


In [0]:
df.write.format('delta').mode('append').saveAsTable('workspace.default.people')

In [0]:
df_old = spark.read.option("versionAsOf", 0).table("people")
df_old.show()
df_new = spark.read.option("versionAsOf", 1).table("people")    
df_new.show()
df_old = spark.read.option("timestampAsOf", "2023-07-10T10:00:00").table("people")
df_old.show()



In [0]:
%sql
RESTORE TABLE people TO VERSION AS OF 0

In [0]:
new_date = [(4, "David", 4000, "India")]
df_new = spark.createDataFrame(new_date, 
                               ["id", "name", "salary", "country"])
df_new.write.format('delta').mode('append').option("mergeSchema", "true").saveAsTable('workspace.default.people')


In [0]:
df_uc = spark.read.table('people')
df_uc.show()

In [0]:
# incremental load 

from delta.tables import *
delta_Table = DeltaTable.forName(spark, "workspace.default.people")

updates = [(2, "Bob", 5000, "India"), (7, "Eve", 6000, "India")]
df_updates = spark.createDataFrame(updates, 
                               ["id", "name", "salary", "country"])

(delta_Table.alias("people").merge(
    df_updates.alias("updates"), "people.id = updates.id"
).whenMatchedUpdate(set={"name": "updates.name", "salary": "updates.salary", "country": "updates.country"})
.whenNotMatchedInsertAll().execute())

df_uc = spark.read.table('people')
df_uc.show()

In [0]:
from delta.tables import *
delta_Table = DeltaTable.forName(spark, "workspace.default.people")

updates = [(2, "Bob", 5000, "India"), (7, "Eve", 6000, "India")]
df_updates = spark.createDataFrame(updates, 
                               ["id", "name", "salary", "country"])

(delta_Table.alias("people").merge(
    df_updates.alias("updates"), "people.id = updates.id"
).whenMatchedUpdate(set={"name": "updates.name", "salary": "updates.salary", "country": "updates.country"})
.whenNotMatchedInsertAll().execute())

df_uc = spark.read.table('people')
df_uc.show()

In [0]:
%sql
ALTER TABLE workspace.default.people SET TBLPROPERTIES (delta.enableChangeDataFeed=true)

In [0]:
# CDC only captures changes AFTER it's enabled
# Get current version before making a change
from delta.tables import *
delta_table = DeltaTable.forName(spark, "workspace.default.people")
history = delta_table.history(1)
current_version = history.select("version").collect()[0][0]
print(f"Current version: {current_version}")

# Make a test change to generate CDC data
test_update = [(1, "Alice", 1500, "USA")]
df_test = spark.createDataFrame(test_update, ["id", "name", "salary", "country"])

(delta_table.alias("people").merge(
    df_test.alias("updates"), "people.id = updates.id"
).whenMatchedUpdate(set={"salary": "updates.salary", "country": "updates.country"})
.execute())

print(f"Change made in version {current_version + 1}")

# Read CDC data starting from the version where the change occurred
df_cdc = spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", current_version + 1).table("people")
print("\nChange Data Feed results:")
df_cdc.show()